# Aarohan-350M — Kaggle SFT (Instruction Fine-Tuning) on TPU v5e-8

**Run AFTER pre-training is complete.**

Fine-tunes **Aarohan-350M** on ~146K instruction-response pairs using all 8 TPU chips
via `xmp.spawn` — the same multi-chip pattern used by the pre-training script.

**Instructions:**
1. Enable TPU: Settings → Accelerator → **TPU v5e-8**
2. Attach your pre-training notebook output (which contains `best.pt`) as input data
3. Add Kaggle Secrets: `WANDB_API_KEY`, `GITHUB_TOKEN`
4. Click **Run All**

**Estimated time:** ~7.5 hours on TPU v5e-8 (fits in one 9-hour session ✅)

In [ ]:
# ── Cell 1: Verify TPU + fix TensorFlow conflict ─────────────
import torch
print(f'PyTorch: {torch.__version__}')

# Remove tensorflow — conflicts with torch_xla on TPU
import subprocess
result = subprocess.run(['pip', 'uninstall', '-y', 'tensorflow'], capture_output=True, text=True)
if 'Successfully' in result.stdout:
    print('tensorflow uninstalled')
subprocess.run(['pip', 'install', 'tensorflow-cpu', '-q'], capture_output=True)
print('tensorflow-cpu installed')

# Verify torch_xla can see the TPU
import os
os.environ.pop('CLOUD_TPU_TASK_ID', None)
os.environ.pop('TPU_PROCESS_ADDRESSES', None)

import torch_xla.core.xla_model as xm
print(f'✅ torch_xla imported successfully (TPU ready for spawn)')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q wandb tokenizers datasets pyyaml

In [ ]:
# ── Cell 3: Clone Aarohan-350M training code ──────────────────
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_REPO = 'Abhik2005/se-llm-data'

if not os.path.exists('/kaggle/working/se-llm-350m'):
    try:
        token    = secrets.get_secret('GITHUB_TOKEN')
        repo_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
    except Exception:
        repo_url = f'https://github.com/{GITHUB_REPO}.git'
    !git clone {repo_url} /kaggle/working/se-llm-350m
else:
    !git -C /kaggle/working/se-llm-350m pull

%cd /kaggle/working/se-llm-350m
!ls -la

In [ ]:
# ── Cell 4: Generate SFT instruction dataset ──────────────────
# Downloads ~146K coding Q&A pairs from HuggingFace (~15 min).
import os

os.makedirs('data/sft', exist_ok=True)

if not os.path.exists('data/sft/sft_data.jsonl'):
    print('Generating SFT dataset from HuggingFace...')
    !python data/sft_data.py
else:
    with open('data/sft/sft_data.jsonl') as f:
        n = sum(1 for _ in f)
    size_mb = os.path.getsize('data/sft/sft_data.jsonl') / 1e6
    print(f'✅ SFT dataset ready: {n:,} samples | {size_mb:.1f} MB')

In [ ]:
# ── Cell 5: Link tokenizer ────────────────────────────────────
import os

os.makedirs('tokenizer', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('checkpoints_sft', exist_ok=True)

# Search for tokenizer in all known Kaggle input paths
tok_candidates = [
    '/kaggle/input/datasets/vedase/se-llm-data/tokenizer.json',
    '/kaggle/input/se-llm-data/tokenizer.json',
]
for tok_src in tok_candidates:
    if os.path.exists(tok_src):
        tok_dest = 'tokenizer/tokenizer.json'
        if not os.path.exists(tok_dest):
            os.symlink(tok_src, tok_dest)
        print(f'✅ Linked tokenizer from {tok_src}')
        break
else:
    print('WARNING: tokenizer.json not found — add the se-llm-data dataset in the sidebar!')

In [ ]:
# ── Cell 6: Load pre-trained Aarohan-350M checkpoint ──────────
# You must attach the OUTPUT of your pre-training notebook session
# as an input dataset in the Kaggle sidebar (+ Add Data → Notebooks).
import os, shutil, glob, torch

# Search all possible locations for best.pt
search_paths = glob.glob('/kaggle/input/**/best.pt', recursive=True)

base_checkpoint = None

if search_paths:
    src  = search_paths[0]
    dest = 'checkpoints/pretrain_best.pt'
    if not os.path.exists(dest):
        print(f'Copying {src} → {dest}  (2.36 GB, please wait...)')
        shutil.copy2(src, dest)
    base_checkpoint = dest

    # Show checkpoint info
    ckpt = torch.load(dest, map_location='cpu', weights_only=False)
    print(f'✅ Pre-trained checkpoint loaded:')
    print(f'   Model:     {ckpt.get("model_config", {}).get("name", "unknown")}')
    print(f'   Step:      {ckpt.get("step", 0):,}')
    print(f'   Tokens:    {ckpt.get("tokens_processed", 0)/1e9:.3f}B')
    print(f'   Val Loss:  {ckpt.get("val_loss", 0):.4f}')
else:
    print('❌ No best.pt found!')
    print('   → In the Kaggle sidebar, click + Add Data → Notebooks')
    print('   → Find your pre-training notebook and add its output')

In [ ]:
# ── Cell 7: Login to W&B ──────────────────────────────────────
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets   = UserSecretsClient()
    wandb_key = secrets.get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('✅ W&B logged in')
except Exception as e:
    print(f'W&B login skipped: {e}')

In [ ]:
# ── Cell 8: RUN INSTRUCTION FINE-TUNING ON TPU ────────────────
# Launches xmp.spawn across all 8 TPU chips.
# Expected: ~7.5 hours | loss should drop from ~3.0 down to ~1.0

assert base_checkpoint is not None, \
    'No checkpoint found! Attach pre-training notebook output in sidebar.'

cmd = f'python training/sft.py --config configs/350m.yaml --base-checkpoint {base_checkpoint}'
print(f'Running: {cmd}\n')
!{cmd}

In [ ]:
# ── Cell 9: Quick quality test ─────────────────────────────────
# Test Aarohan-350M as a chat assistant!
import torch, os, sys, glob
sys.path.insert(0, '/kaggle/working/se-llm-350m')

from evaluation.generate import load_model_from_checkpoint, load_tokenizer, chat_turn

# Use CPU for inference test (no need for full TPU)
device = torch.device('cpu')

# Find the SFT checkpoint
ckpt_path = 'checkpoints_sft/sft_final.pt'
if not os.path.exists(ckpt_path):
    pts = sorted(glob.glob('checkpoints_sft/*.pt'))
    ckpt_path = pts[-1] if pts else None

if ckpt_path and os.path.exists(ckpt_path):
    model, cfg = load_model_from_checkpoint(ckpt_path, device)
    tokenizer  = load_tokenizer('tokenizer/tokenizer.json')

    test_prompts = [
        'Write a Python function to check if a number is prime.',
        'What is the difference between a stack and a queue?',
        'Write a SQL query to find the top 3 most expensive products.',
    ]

    for prompt in test_prompts:
        print(f'\n{"─"*55}')
        print(f'User: {prompt}')
        response = chat_turn(model, tokenizer, prompt, device=device, max_new_tokens=150)
        print(f'Aarohan:\n{response}')

    print('\n✅ Aarohan-350M SFT quality check complete!')
else:
    print('No SFT checkpoint found — check if training completed successfully.')

In [ ]:
# ── Cell 10: Done! ────────────────────────────────────────────
import glob, os

print('SFT checkpoints in /kaggle/working/se-llm-350m/checkpoints_sft/')
for ckpt in sorted(glob.glob('checkpoints_sft/*.pt')):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'  {os.path.basename(ckpt):40s}  {size_mb:.0f} MB')

print()
print('✅ Aarohan-350M SFT complete!')
print('   Download sft_final.pt and test locally with:')
print('   python evaluation/generate.py \\')
print('     --checkpoint checkpoints_sft/sft_final.pt \\')
print('     --mode chat')